In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from utils.metric import metric
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb
import lightgbm as lgb

np.random.seed(42)
pd.set_option("display.max_columns", 100)

In [2]:
HORIZONS = [12, 24, 48, 72]

In [3]:
train = pd.read_csv('../../data/train.csv')

In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    eps = 1e-6

    dist = d["dist_min_ci_0_5h"].astype(float).clip(lower=1.0)  # meters
    dist_km = dist / 1000.0

    perims = d["num_perimeters_0_5h"].astype(float).clip(lower=0.0)
    dt_span = d["dt_first_last_0_5h"].astype(float).clip(lower=0.0)

    closing = d["closing_speed_m_per_h"].astype(float)
    closing_pos = closing.clip(lower=0.0)

    radial_rate = d["radial_growth_rate_m_per_h"].astype(float).clip(lower=0.0)
    along_track = d["along_track_speed"].astype(float) if "along_track_speed" in d.columns else 0.0

    align_abs = d["alignment_abs"].astype(float).clip(lower=0.0, upper=1.0)
    align_cos = d["alignment_cos"].astype(float) if "alignment_cos" in d.columns else 0.0

    area = d["area_first_ha"].astype(float).clip(lower=0.0)
    growth_rate_ha_h = d["area_growth_rate_ha_per_h"].astype(float).clip(lower=0.0)

    d["dist_km"] = dist_km
    d["log_dist"] = np.log1p(dist)
    d["sqrt_dist"] = np.sqrt(dist)

    d["inv_dist"] = 1.0 / (dist + 1.0)
    d["inv_dist_sq"] = d["inv_dist"] ** 2
    d["inv_dist_km"] = 1.0 / (dist_km + 0.05)

    d["zone_lt3km"] = (dist < 3000).astype(int)
    d["zone_3to5km"] = ((dist >= 3000) & (dist < 5000)).astype(int)
    d["zone_5to10km"] = ((dist >= 5000) & (dist < 10000)).astype(int)
    d["zone_ge10km"] = (dist >= 10000).astype(int)

    if "dist_fit_r2_0_5h" in d.columns:
        r2 = d["dist_fit_r2_0_5h"].astype(float).clip(lower=0.0, upper=1.0)
        d["dist_trend_reliable"] = (r2 > 0.6).astype(int)
        d["dist_r2"] = r2
    else:
        d["dist_trend_reliable"] = 0
        d["dist_r2"] = 0.0

    for col in ["dist_std_ci_0_5h", "dist_change_ci_0_5h", "dist_slope_ci_0_5h", "dist_accel_m_per_h2"]:
        if col in d.columns:
            d[col] = d[col].astype(float)

    if "dist_change_ci_0_5h" in d.columns:
        d["dist_change_km"] = d["dist_change_ci_0_5h"] / 1000.0
        d["dist_change_norm"] = d["dist_change_ci_0_5h"] / (dist + 1.0)

    if "dist_slope_ci_0_5h" in d.columns:
        d["dist_slope_norm"] = d["dist_slope_ci_0_5h"] / (dist + 1.0)

    if "dist_std_ci_0_5h" in d.columns:
        d["dist_std_norm"] = d["dist_std_ci_0_5h"] / (dist + 1.0)

    effective_closing = closing_pos + radial_rate
    d["effective_closing_speed"] = effective_closing

    d["eta_closing_h"] = np.where(closing_pos > 0.1, dist / (closing_pos + eps), 9999.0)
    d["eta_effective_h"] = np.where(effective_closing > 0.1, dist / (effective_closing + eps), 9999.0)

    d["log_eta_effective"] = np.log1p(np.clip(d["eta_effective_h"], 0, 9999))
    d["log_eta_closing"] = np.log1p(np.clip(d["eta_closing_h"], 0, 9999))

    for h in HORIZONS:
        d[f"eta_within_{h}h"] = (d["eta_effective_h"] <= float(h)).astype(int)
        d[f"eta_margin_{h}h"] = d["eta_effective_h"] - float(h)

    if isinstance(along_track, pd.Series):
        along_pos = along_track.astype(float).clip(lower=0.0)
        d["eta_alongtrack_h"] = np.where(along_pos > 0.1, dist / (along_pos + eps), 9999.0)
        d["log_eta_alongtrack"] = np.log1p(np.clip(d["eta_alongtrack_h"], 0, 9999))
    else:
        d["eta_alongtrack_h"] = 9999.0
        d["log_eta_alongtrack"] = np.log1p(9999.0)

    fire_radius_m = np.sqrt((area * 10000.0) / np.pi)
    d["fire_radius_m"] = fire_radius_m
    d["radius_to_dist"] = fire_radius_m / (dist + 1.0)

    d["dist_minus_radius"] = np.clip(dist - fire_radius_m, 1.0, None)
    d["log_dist_minus_radius"] = np.log1p(d["dist_minus_radius"])

    d["area_to_dist_km"] = area / (dist_km + 0.1)
    d["growth_to_dist_km"] = growth_rate_ha_h / (dist_km + 0.1)

    if "radial_growth_m" in d.columns:
        d["radial_growth_m"] = d["radial_growth_m"].astype(float).clip(lower=0.0)
        d["radial_to_dist"] = d["radial_growth_m"] / (dist + 1.0)

    d["threat_pressure"] = align_abs * effective_closing / (np.log1p(dist) + eps)
    d["alignment_x_closing"] = align_abs * closing_pos
    d["alignment_x_effective"] = align_abs * effective_closing

    if "cross_track_component" in d.columns:
        ctc = d["cross_track_component"].astype(float)
        d["cross_track_abs"] = np.abs(ctc)
        d["cross_track_norm"] = np.abs(ctc) / (dist + 1.0)

    if "spread_bearing_sin" in d.columns and "spread_bearing_cos" in d.columns:
        sb_sin = d["spread_bearing_sin"].astype(float)
        sb_cos = d["spread_bearing_cos"].astype(float)
        d["bearing_strength"] = np.sqrt(sb_sin**2 + sb_cos**2) 

    d["has_movement"] = (perims > 1).astype(int)
    d["perim_density"] = perims / (dt_span + 0.25) 
    d["short_window"] = (dt_span < 0.5).astype(int)

    if "low_temporal_resolution_0_5h" in d.columns:
        d["low_temporal_resolution_0_5h"] = d["low_temporal_resolution_0_5h"].astype(int)

    hour = d["event_start_hour"].astype(float) if "event_start_hour" in d.columns else 0.0
    d["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    d["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)

    dow = d["event_start_dayofweek"].astype(float) if "event_start_dayofweek" in d.columns else 0.0
    d["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    d["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)

    month = d["event_start_month"].astype(float) if "event_start_month" in d.columns else 1.0
    d["month_sin"] = np.sin(2 * np.pi * month / 12.0)
    d["month_cos"] = np.cos(2 * np.pi * month / 12.0)

    d = d.replace([np.inf, -np.inf], np.nan).fillna(0)

    return d

In [5]:
def make_strata(t, e):
    strata = np.zeros(len(t), dtype=int)
    
    strata[e == 0] = 0
    
    strata[(e == 1) & (t <= 24)] = 1
    
    strata[(e == 1) & (t > 24) & (t <= 48)] = 2
    
    strata[(e == 1) & (t > 48)] = 3
    
    return strata

In [6]:
train_fe = build_features(train)

In [7]:
def enforce_monotone(P):
    P = np.clip(P, 0.0, 1.0)
    P[:, 1] = np.maximum(P[:, 1], P[:, 0])
    P[:, 2] = np.maximum(P[:, 2], P[:, 1])
    P[:, 3] = np.maximum(P[:, 3], P[:, 2])
    return np.clip(P, 0.0, 1.0)

def masked_brier_for_horizon(p, t, e, H):
    mask = []
    y = []
    for i in range(len(t)):
        if e[i] == 1 and t[i] <= H:
            mask.append(True); y.append(1)
        elif t[i] > H:
            mask.append(True); y.append(0)
        else:
            mask.append(False); y.append(0)
    mask = np.array(mask, dtype=bool)
    y = np.array(y, dtype=int)
    if mask.sum() == 0:
        return 0.0
    return np.mean((p[mask] - y[mask])**2)

In [8]:
drop_cols = ["event_id", "event", "time_to_hit_hours"]

X = train_fe.drop(columns=drop_cols)
t = train_fe["time_to_hit_hours"].values.astype(float)
e = train_fe["event"].values.astype(int)

strata = make_strata(t, e)

In [9]:
def km_censor_survival(times, censor_event):
    order = np.argsort(times)
    times = times[order]
    censor_event = censor_event[order]

    uniq = np.unique(times)
    n = len(times)

    at_risk = n
    G = 1.0
    G_map = {}

    for tt in uniq:
        m = np.sum(times == tt)
        d = np.sum((times == tt) & (censor_event == 1))

        if at_risk > 0:
            G *= (1.0 - d / at_risk)

        G_map[tt] = max(G, 1e-6)
        at_risk -= m

    uniq_times = np.array(sorted(G_map.keys()))
    G_vals = np.array([G_map[u] for u in uniq_times])
    return uniq_times, G_vals


def G_of_t(t_query, uniq_times, G_vals):
    t_query = np.asarray(t_query)
    idx = np.searchsorted(uniq_times, t_query, side="right") - 1
    out = np.ones_like(t_query, dtype=float)
    ok = idx >= 0
    out[ok] = G_vals[idx[ok]]
    return np.clip(out, 1e-6, 1.0)

In [10]:
def make_ipcw_data(X, t, e, H, uniq_times, G_vals, weight_cap=30.0):

    y = ((e == 1) & (t <= H)).astype(int)

    mask = ((e == 1) & (t <= H)) | (t > H)

    t_clip = np.minimum(t, H)

    G_t = G_of_t(t_clip, uniq_times, G_vals)

    w = 1.0 / G_t
    w = np.clip(w, 1.0, weight_cap)

    return X[mask], y[mask], w[mask], mask

In [11]:
def bagged_lgb_predict(X_tr, y_tr, w_tr, X_va, n_seeds=5):

    preds = np.zeros(len(X_va))

    for seed in range(42, 42 + n_seeds):
        model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.02,
            num_leaves=15,
            max_depth=3,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=8.0,
            objective="binary",
            random_state=seed,
            verbose=-1
        )

        model.fit(X_tr, y_tr, sample_weight=w_tr)
        preds += model.predict_proba(X_va)[:, 1]

    return preds / n_seeds

In [12]:
def make_tail_data(X, t, e, H0, H1, uniq_times, G_vals, weight_cap=50.0):
    y = ((e == 1) & (t > H0) & (t <= H1)).astype(int)

    mask = ((e == 1) & (t > H0) & (t <= H1)) | (t > H1)

    t_clip = np.minimum(t, H1)
    G_t = G_of_t(t_clip, uniq_times, G_vals)
    w = 1.0 / G_t
    w = np.clip(w, 1.0, weight_cap)

    return X[mask], y[mask], w[mask], mask

In [13]:
def run_pipeline(X_tr, t_tr, e_tr, X_va, t_va, e_va):

    # ---------------- KM ----------------
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    # ================= 12H =================
    H = 12
    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
        X_tr, t_tr, e_tr, H, uniq_times, G_vals
    )
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(
        X_va, t_va, e_va, H, uniq_times, G_vals
    )

    p12_full = np.zeros(len(X_va))
    p12_full[mask_va] = bagged_lgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=10)
    # ================= 24H =================
    H = 24
    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
        X_tr, t_tr, e_tr, H, uniq_times, G_vals
    )
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(
        X_va, t_va, e_va, H, uniq_times, G_vals
    )

    p24_full = np.zeros(len(X_va))
    p24_full[mask_va] = bagged_lgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=10)

    X48_tr, y48_tr, w48_tr, m48_tr = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
    X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

    tail_feats = [
        "dist_km",
        "eta_effective_h",
        "effective_closing_speed",
        "alignment_abs",
        "has_movement"
    ]

    lr_48 = LogisticRegression(
        C=0.2,
        solver="lbfgs",
        max_iter=2000
    )

    lr_48.fit(
        X48_tr[tail_feats],
        y48_tr,
        sample_weight=w48_tr
    )

    p48_tail_full = np.zeros(len(X_va))
    p48_tail_full[m48_va] = lr_48.predict_proba(
        X48_va[tail_feats]
    )[:, 1]

    p48_full = p24_full + (1 - p24_full) * p48_tail_full

    # ================= 48–72 Tail =================
    late_events = np.sum((e_tr == 1) & (t_tr > 48))
    survive_48 = np.sum(t_tr > 48)
    alpha = (late_events + 1) / (survive_48 + 2)

    p72 = p48_full + (1 - p48_full) * alpha

    # ================= Final P =================
    P = np.zeros((len(X_va), 4))
    P[:, 0] = p12_full
    P[:, 1] = p24_full
    P[:, 2] = p48_full
    P[:, 3] = p72

    P = enforce_monotone(P)

    b12 = masked_brier_for_horizon(P[:,0], t_va, e_va, 12)
    b24 = masked_brier_for_horizon(P[:,1], t_va, e_va, 24)
    b48 = masked_brier_for_horizon(P[:,2], t_va, e_va, 48)
    b72 = masked_brier_for_horizon(P[:,3], t_va, e_va, 72)

    print("Brier 12:", b12)
    print("Brier 24:", b24)
    print("Brier 48:", b48)
    print("Brier 72:", b72)

    return metric(P, t_va, e_va), P

In [14]:
def bagged_xgb_predict(X_tr, y_tr, w_tr, X_va, n_seeds=5):
    preds = np.zeros(len(X_va))

    for seed in range(42, 42 + n_seeds):
        model = xgb.XGBClassifier(
                n_estimators=2500,
                learning_rate=0.03,
                max_depth=8,
                min_child_weight=1,
                subsample=0.9,
                colsample_bytree=0.6,  # lower column sampling
                reg_alpha=0.5,
                reg_lambda=1.0,
                gamma=2.0,  # force split quality
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=seed,
        )
        model.fit(X_tr, y_tr, sample_weight=w_tr)
        preds += model.predict_proba(X_va)[:, 1]

    return preds / n_seeds


def run_pipeline_xgb(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    # ---------------- KM ----------------
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    # ================= 12H =================
    H = 12
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)

    p12_full = np.zeros(len(X_va))
    p12_full[mask_va] = bagged_xgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=8)

    # ================= 24H =================
    H = 24
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)

    p24_full = np.zeros(len(X_va))
    p24_full[mask_va] = bagged_xgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=8)

    # ================= 24–48 Tail =================
    X48_tr, y48_tr, w48_tr, _ = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
    X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

    # use same tail_feats you already defined (or expand)
    tail_feats = [
        "dist_km",
        "eta_effective_h",
        "effective_closing_speed",
        "alignment_abs",
        "has_movement"
    ]

    p48_tail_full = np.zeros(len(X_va))
    p48_tail_full[m48_va] = bagged_xgb_predict(
        X48_tr[tail_feats], y48_tr, w48_tr,
        X48_va[tail_feats],
        n_seeds=10
    )

    p48_full = p24_full + (1 - p24_full) * p48_tail_full

    # ================= 48–72 Tail (same alpha trick as you) =================
    late_events = np.sum((e_tr == 1) & (t_tr > 48))
    survive_48 = np.sum(t_tr > 48)
    alpha = (late_events + 1) / (survive_48 + 2)

    p72 = p48_full + (1 - p48_full) * alpha

    # ================= Final P =================
    P = np.zeros((len(X_va), 4))
    P[:, 0] = p12_full
    P[:, 1] = p24_full
    P[:, 2] = p48_full
    P[:, 3] = p72

    P = enforce_monotone(P)

    return metric(P, t_va, e_va), P

In [15]:
def run_pipeline_lgb_wrapper(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    (c, b, s), P = run_pipeline(X_tr, t_tr, e_tr, X_va, t_va, e_va)
    return (c, b, s), P


def collect_oof_preds(X, t, e, strata, skf):
    oof_lgb = np.zeros((len(X), 4))
    oof_xgb = np.zeros((len(X), 4))

    fold_scores_lgb = []
    fold_scores_xgb = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, strata)):
        print("Fold:", fold)

        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        t_tr, t_va = t[tr_idx], t[va_idx]
        e_tr, e_va = e[tr_idx], e[va_idx]

        (c_l, b_l, s_l), P_lgb = run_pipeline_lgb_wrapper(X_tr, t_tr, e_tr, X_va, t_va, e_va)
        (c_x, b_x, s_x), P_xgb = run_pipeline_xgb(X_tr, t_tr, e_tr, X_va, t_va, e_va)

        oof_lgb[va_idx] = P_lgb
        oof_xgb[va_idx] = P_xgb

        fold_scores_lgb.append(s_l)
        fold_scores_xgb.append(s_x)

        print("LGB fold score:", s_l)
        print("XGB fold score:", s_x)

    print("LGB CV:", np.mean(fold_scores_lgb), np.std(fold_scores_lgb))
    print("XGB CV:", np.mean(fold_scores_xgb), np.std(fold_scores_xgb))
    return oof_lgb, oof_xgb

In [16]:
def blend_global_weight(P_a, P_b, w):
    P = w * P_a + (1 - w) * P_b
    return enforce_monotone(P)

def find_best_global_blend(oof_lgb, oof_xgb, t, e):
    best = (-1e9, None)  # (score, w)
    for w in np.linspace(0.0, 1.0, 41):  # step 0.025
        P = blend_global_weight(oof_lgb, oof_xgb, w)
        c, b, s = metric(P, t, e)
        if s > best[0]:
            best = (s, w)
    print("Best global blend:", best[1], "OOF score:", best[0])
    return best[1]

def blend_per_horizon(P_a, P_b, w_vec):
    P = np.zeros_like(P_a)
    for j in range(4):
        P[:, j] = w_vec[j] * P_a[:, j] + (1 - w_vec[j]) * P_b[:, j]
    return enforce_monotone(P)

def find_best_per_horizon_blend(oof_lgb, oof_xgb, t, e):
    # coarse grid to avoid overfitting and keep it fast
    grid = np.linspace(0.0, 1.0, 11)  # step 0.1
    best = (-1e9, None)

    for w0 in grid:
        for w1 in grid:
            for w2 in grid:
                for w3 in grid:
                    w_vec = np.array([w0, w1, w2, w3])
                    P = blend_per_horizon(oof_lgb, oof_xgb, w_vec)
                    c, b, s = metric(P, t, e)
                    if s > best[0]:
                        best = (s, w_vec.copy())

    print("Best per-horizon blend:", best[1], "OOF score:", best[0])
    return best[1]

In [17]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb, oof_xgb = collect_oof_preds(X, t, e, strata, skf)

# 1) safest: global blend
w_best = find_best_global_blend(oof_lgb, oof_xgb, t, e)
P_blend = blend_global_weight(oof_lgb, oof_xgb, w_best)
print("Blended OOF metric:", metric(P_blend, t, e))
print("LGB OOF metric:", metric(oof_lgb, t, e))
print("XGB OOF metric:", metric(oof_xgb, t, e))

# 2) optional: per-horizon blend (try only if global helped)
# w_vec_best = find_best_per_horizon_blend(oof_lgb, oof_xgb, t, e)
# P_blend2 = blend_per_horizon(oof_lgb, oof_xgb, w_vec_best)
# print("Per-horizon blended OOF metric:", metric(P_blend2, t, e))

/home/rishik/dev/personal/challenges/wids/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Fold: 0
Brier 12: 0.07497730518808951
Brier 24: 0.03110909375289811
Brier 48: 0.008622249614094732
Brier 72: 0.010741705111247177
LGB fold score: 0.9678126463880337
XGB fold score: 0.9688115967975963
Fold: 1
Brier 12: 0.05812968180426433
Brier 24: 0.03445326451969999
Brier 48: 0.044581757436999166
Brier 72: 0.01124067933464
LGB fold score: 0.953575829970009
XGB fold score: 0.9505742224549633
Fold: 2
Brier 12: 0.040904085286378004
Brier 24: 0.021023204433945366
Brier 48: 0.035370030195871274
Brier 72: 0.019279700965971067
LGB fold score: 0.9650122827962151
XGB fold score: 0.9616799757653933
Fold: 3
Brier 12: 0.08579725770578515
Brier 24: 0.04357956024619748
Brier 48: 0.02016054409912897
Brier 72: 0.050524840884399076
LGB fold score: 0.9496647566526409
XGB fold score: 0.9647731126605148
Fold: 4
Brier 12: 0.04841592395990946
Brier 24: 0.021231797435158614
Brier 48: 0.005001727158377667
Brier 72: 0.011887527314678159
LGB fold score: 0.9766444581981886
XGB fold score: 0.9712980070154831
LGB